In [6]:
"""
Greedy algorithms typically have this top-down design:
make a choise and then solve a subproblem, rather than
the bottom-up technique of solving subproblems before making
a choice.

stores the indexes of the
start time of candidate ≥ finish time of last selected

activity_selector example:
s = [0, 1, 3, 0, 5, 3, 5, 6, 8, 8, 2, 12]
f = [0, 4, 5, 6, 7, 9, 9, 10, 11, 12, 14, 16]
[1] + [4] + [8] + [11] + []
1 > 0, 5 > 4, 8 > 7, 12 > 11

simply picks times that don't overlap
"""
def recursive_activity_selector(s, f, k):
    m = k + 1
    
    while m < len(s) and s[m] < f[k]:
        m += 1
    
    if m < len(s):
        return [m] + recursive_activity_selector(s, f, m)
    else:
        return []

In [4]:
def greedy_activity_selector(s, f):
    selected = [0]
    k = 0
    
    for m in range(1, len(s)):
        if s[m] >= f[k]:
            selected.append(m)
            k = m
            
    return selected

In [5]:
def greedy_activity_selector_clean(activities):
    activities = sorted(activities, key=lambda x: x[1])
    
    selected = [activities[0]]
    last_finish = activities[0][1]
    
    for start, finish in activities[1:]:
        if start >= last_finish:
            selected.append((start, finish))
            last_finish = finish
            
    return selected

In [7]:
"""
The DP solution proves correctness structure:
Optimal solution = optimal left + optimal right + chosen activity

The greedy solution exploits a structural property:
Choosing the activity that finishes earliest always leaves maximum room for the rest.

That property collapses O(n³) → O(n).
The maximum number of mutually compatible activities we can select
c[i][j] = maximum number of non-overlapping activities
          that fit strictly between activity i and activity j

Imagine this:
i -------------------- j

We’re trying to pack activities between those boundaries.
For each possible choice k:
i ---- k ---- j

We split into two independent subproblems:
Left side: (i, k)
Right side: (k, j)
Then combine:
best_left + best_right + 1

So the DP table stores:
The size of the best schedule in every possible sub-interval.
"""
def activity_selector_dp(s, f):
    n = len(s)
    
    c = [[0] * n for _ in range(n)]
    solution = [[None] * n for _ in range(n)]
    
    # gap = distance between i and j
    for gap in range(2, n):
        for i in range(n - gap):
            j = i + gap
            
            for k in range(i + 1, j):
                if f[i] <= s[k] and f[k] <= s[j]:
                    value = c[i][k] + c[k][j] + 1
                    
                    if value > c[i][j]:
                        c[i][j] = value
                        solution[i][j] = k
    
    return c, solution

In [8]:
def reverse_greedy_activity_selector(s, f):
    n = len(s)
    selected = [n - 1]   # pick activity that starts last
    k = n - 1
    
    for m in range(n - 2, -1, -1):
        if f[m] <= s[k]:
            selected.append(m)
            k = m
            
    return selected[::-1]

In [9]:
"""
Minimum lecture halls needed: 3

Assignments:
Activity (1, 4) -> Hall 1
Activity (2, 5) -> Hall 2
Activity (3, 6) -> Hall 3
Activity (4, 7) -> Hall 1

Activities (sorted by start time):
A (1,4)
B (2,5)
C (3,6)
D (4,7)
------------------------------------

Activity A (1,4)
Heap empty → open Hall 1.
Heap:
[(4, H1)]

------------------------------------

Activity B (2,5)
Earliest finish = 4
4 > 2 → Hall 1 not free.
Open Hall 2.
Heap:
[(4, H1), (5, H2)]

------------------------------------

Activity C (3,6)
Earliest finish = 4
4 > 3 → no hall free.
Open Hall 3.
Heap:
[(4, H1), (5, H2), (6, H3)]

------------------------------------

Activity D (4,7)
Earliest finish = 4
4 <= 4 → Hall 1 becomes free.
Reuse Hall 1.
Remove (4, H1)
Insert (7, H1)
Heap:
[(5, H2), (6, H3), (7, H1)]

------------------------------------

Hall 1: A (1,4), D (4,7)
Hall 2: B (2,5)
Hall 3: C (3,6)

"""
import heapq

def interval_partitioning(activities):
    # activities: list of (start, finish)
    
    # Step 1: sort by start time
    activities.sort(key=lambda x: x[0])
    
    heap = []  # min-heap of (finish_time, hall_number)
    hall_count = 0
    assignment = []
    
    for start, finish in activities:
        # If there's a hall that becomes free
        if heap and heap[0][0] <= start:
            free_finish, hall = heapq.heappop(heap)
        else:
            hall_count += 1
            hall = hall_count
        
        assignment.append((start, finish, hall))
        heapq.heappush(heap, (finish, hall))
    
    return hall_count, assignment

In [10]:
"""
activities: list of (start, finish, value)
"""
import bisect

def weighted_activity_selection(activities):
    # Step 1: sort by finish time
    activities = sorted(activities, key=lambda x: x[1])
    
    n = len(activities)
    
    starts = [a[0] for a in activities]
    finishes = [a[1] for a in activities]
    values = [a[2] for a in activities]
    
    # Step 2: compute p array
    p = [0] * n
    for j in range(n):
        # find rightmost activity that finishes <= starts[j]
        i = bisect.bisect_right(finishes, starts[j]) - 1
        p[j] = i
    
    # Step 3: DP
    OPT = [0] * (n + 1)
    
    for j in range(1, n + 1):
        include = values[j - 1] + (OPT[p[j - 1] + 1] if p[j - 1] != -1 else 0)
        exclude = OPT[j - 1]
        OPT[j] = max(include, exclude)
    
    return OPT[n]

In [11]:
"""
Greedy works all the time if the problem has 2 properties:
1) greedy-choice property
2) optimal substructure
"""

'\nGreedy works all the time if the problem has 2 properties:\n1) greedy-choice property\n2) optimal substructure\n'

In [12]:
"""
Optimal choice:
Item 2 (value 100, weight 20)
Item 3 (value 120, weight 30)
Total value = 220

0-1 knapsack works with DP but not greedy
"""
def knapsack(values, weights, W):
    n = len(values)
    
    dp = [[0 for _ in range(W + 1)] for _ in range(n + 1)]

    for i in range(1, n + 1):
        for w in range(W + 1):
            if weights[i-1] <= w:
                dp[i][w] = max(
                    dp[i-1][w], 
                    values[i-1] + dp[i-1][w - weights[i-1]]
                )
            else:
                dp[i][w] = dp[i-1][w]

    return dp[n][W]

values = [60, 100, 120]
weights = [10, 20, 30]
W = 50

print(knapsack(values, weights, W))

220


In [13]:
"""
We iterate backwards so the same item isn't reused
multiple times (which would turn it into a fractional/unbounded knapsack).

Instead of storing rows for each item, it reuses the same array.

Learning / debugging / reconstructing items → use full table
Large problems where memory matters → use space-optimized
"""
def knapsack_optimized_space(values, weights, W):
    n = len(values)
    dp = [0] * (W + 1)

    for i in range(n):
        for w in range(W, weights[i] - 1, -1):
            dp[w] = max(dp[w], values[i] + dp[w - weights[i]])

    return dp[W]

In [14]:
"""
Greedy Algorithm

Normally knapsack is hard because light items might have low value and heavy items might have high value.

Sort items by increasing weight (already consistent with decreasing value).
Add items to the knapsack in that order until the next item would exceed capacity.
"""
def knapsack_variant(values, weights, W):
    n = len(values)
    
    total_value = 0
    total_weight = 0
    chosen = []

    for i in range(n):
        if total_weight + weights[i] <= W:
            total_weight += weights[i]
            total_value += values[i]
            chosen.append(i)

    return total_value, chosen

In [15]:
"""
This covers the leftmost uncovered point and as many points to its right as possible.
0.2, 0.7, 1.4, 1.6, 2.1

start = 0.2
interval = [0.2, 1.2]
Covers:
0.2, 0.7

start = 1.4
interval = [1.4, 2.4]
Covers:
1.4, 1.6, 2.1
"""
def cover_points(points):
    points.sort()
    intervals = []

    i = 0
    n = len(points)

    while i < n:
        start = points[i]
        end = start + 1
        intervals.append((start, end))

        i += 1
        # Skip points covered by this interval
        while i < n and points[i] <= end:
            i += 1

    return intervals

In [17]:
"""
using builtin python implementation of priority queue
for use in greedy algorithms
"""
import heapq

def extract_min(c):
    return heapq.heappop(c)

def insert(c, z):
    heapq.heappush(c, z)

In [18]:
class Node:
    def __init__(self, freq, left=None, right=None):
        self.freq = freq
        self.left = left
        self.right = right

    def __lt__(self, other):
        return self.freq < other.freq

In [20]:
"""
Symbol	Frequency
A	5
B	9
C	12
D	13
E	16
F	45

[5(A), 9(B), 12(C), 13(D), 16(E), 45(F)]

Step 1
Extract the two smallest.
x = 5(A)
y = 9(B)

Create new node:
z = 14
z.left = A
z.right = B

Tree created:
   14
  /  \
 A    B
 5    9

Insert back into queue.
[12(C), 13(D), 14, 16(E), 45(F)]
etc

             100
            /   \
          F      55
         45     /  \
              25    30
             / \   /  \
            C  D  14   E
           12 13 / \
                 A  B
                 5  

big frequencies → near root (short codes)
small frequencies → deeper leaves (long codes)
              root
             /    \
           ...    ...
          /          \
        x(5)        ...
                    /   \
                 a(20)  b(25)
We swap their positions in the tree (not their frequencies).

              root
             /    \
           ...    ...
          /          \
        a(20)       ...
                    /   \
                 x(5)   b(25)

Now the smaller frequency is deeper, which is better for compression.

Why the Cost Cannot Increase
Cost of a coding tree:
Total cost = Σ frequency × depth

Before swap:
x.freq * depth(x) + a.freq * depth(a)

After swap:
x.freq * depth(a) + a.freq * depth(x)

Difference:
(depth(a) - depth(x)) * (x.freq - a.freq)

Since
depth(a) ≥ depth(x)
x.freq ≤ a.freq
the result is ≤ 0.
"""
def huffman(c):
    for i in range(len(c) - 1):
        x = extract_min(c)
        y = extract_min(c)

        z = Node(x.freq + y.freq)
        z.left = x
        z.right = y

        insert(c, z)

    return extract_min(c)

In [23]:
"""
Any binary tree representing an optimal prefix-free code must be a full binary tree.
Thus optimal prefix codes always correspond to full binary trees, which is why Huffman trees are always full.
"""

'\nAny binary tree representing an optimal prefix-free code must be a full binary tree.\nThus optimal prefix codes always correspond to full binary trees, which is why Huffman trees are always full.\n'

In [24]:
"""
Symbol | Code |	Length
h	       0	  1
g	      10	  2
f	     110 	  3
e	    1110	  4
d	   11110	  5
c	  111110	  6
a    1111110	  7
b    1111111	  7

1(a), 1(b), 2(c), 3(d), 5(e), 8(f), 13(g), 21(h)

            h
             \
              g
               \
                f
                 \
                  e
                   \
                    d
                     \
                      c
                     / \
                    a   b

B(T) = Σ (left_child_freq + right_child_freq) for every internal node
          (39)
         /    \
      (14)    (25)
      /  \    /  \
    A(5) B(9) C(12) D(13)

Symbol	fᵢ	dᵢ	fᵢdᵢ
A	    5	2	 10
B	    9	2	 18
C	    12	2	 24
D	    13	2	 26

Total cost:
10 + 18 + 24 + 26 = 78
-------------------------------------
Node (14)
A(5) + B(9) = 14
Node (25)
C(12) + D(13) = 25
Root
14 + 25 = 39
Add them:
14 + 25 + 39 = 78
SAME!

Level 0: 39
Level 1: 14 + 25
Level 2: 5 + 9 + 12 + 13
"""

'\nSymbol | Code |\tLength\nh\t       0\t  1\ng\t      10\t  2\nf\t     110 \t  3\ne\t    1110\t  4\nd\t   11110\t  5\nc\t  111110\t  6\na    1111110\t  7\nb    1111111\t  7\n\n1(a), 1(b), 2(c), 3(d), 5(e), 8(f), 13(g), 21(h)\n\n            h\n                           g\n                               f\n                                   e\n                                       d\n                                           c\n                     /                     a   b\n\nB(T) = Σ (left_child_freq + right_child_freq) for every internal node\n          (39)\n         /          (14)    (25)\n      /  \\    /      A(5) B(9) C(12) D(13)\n\nSymbol\tfᵢ\tdᵢ\tfᵢdᵢ\nA\t    5\t2\t 10\nB\t    9\t2\t 18\nC\t    12\t2\t 24\nD\t    13\t2\t 26\n\nTotal cost:\n10 + 18 + 24 + 26 = 78\n-------------------------------------\nNode (14)\nA(5) + B(9) = 14\nNode (25)\nC(12) + D(13) = 25\nRoot\n14 + 25 = 39\nAdd them:\n14 + 25 + 39 = 78\nSAME!\n\nLevel 0: 39\nLevel 1: 14 + 25\nLevel 2: 5 + 9 + 12 + 

In [25]:
"""
C = []               # initial cache
k = 3                # cache capacity
requests = [1,2,3,2,4,1,5,2,1,2,3,4]

furthest_in_future_cache(C, k, requests)
"""
def furthest_in_future_cache(C, k, requests):
    cache = list(C)

    if len(cache) > k:
        cache = cache[:k]

    n = len(requests)

    for i in range(n):
        block = requests[i]

        if block in cache:
            print(f"Request {block}: Cache HIT")
            continue

        print(f"Request {block}: Cache MISS")

        if len(cache) < k:
            cache.append(block)
            print("  No eviction (cache had space)")
            continue

        # Determine which block to evict
        furthest_index = -1
        block_to_evict = None
        for c in cache:
            try:
                next_use = requests.index(c, i + 1)
            except ValueError:
                next_use = float('inf')  # never used again

            if next_use > furthest_index:
                furthest_index = next_use
                block_to_evict = c

        # Evict and insert
        cache.remove(block_to_evict)
        cache.append(block)

        if furthest_index == float('inf'):
            print(f"  Evicted {block_to_evict} (never used again)")
        else:
            print(f"  Evicted {block_to_evict} (used again at index {furthest_index})")

        print(f"  Cache state: {cache}")

In [29]:
"""
LRU evicts blocks that haven't been used recently, but sometimes those blocks are needed again very soon, while other blocks that were used recently won’t be needed for a long time.
The optimal algorithm avoids this by evicting the block whose next use is farthest in the future.

Croesus's version assumes the optimal solution already matches the greedy algorithm,
but the real proof must handle the case where they differ and show we can transform
the optimal solution to match the greedy choice.

If the optimal solution evicts y while greedy evicts x,
we show we can modify the optimal solution so that it evicts x
without increasing misses. Therefore an optimal solution exists that matches the greedy choice.
"""

"\nLRU evicts blocks that haven't been used recently, but sometimes those blocks are needed again very soon, while other blocks that were used recently won’t be needed for a long time.\nThe optimal algorithm avoids this by evicting the block whose next use is farthest in the future.\n\nCroesus's version assumes the optimal solution already matches the greedy algorithm,\nbut the real proof must handle the case where they differ and show we can transform\nthe optimal solution to match the greedy choice.\n\nIf the optimal solution evicts y while greedy evicts x,\nwe show we can modify the optimal solution so that it evicts x\nwithout increasing misses. Therefore an optimal solution exists that matches the greedy choice.\n"

In [30]:
"""
greedy fails when coin system is different than US:
Consider denominations:
{1,3,4}

Find change for:
n=6

Take 4
Remaining 2
Take 1 + 1
Coins used:
4+1+1 = 3 coins

Optimal Solution
3+3 = 2 coins
"""
def greedy_change(n):
    q = n // 25
    n = n % 25

    d = n // 10
    n = n % 10

    k = n // 5
    n = n % 5

    p = n

    return (q, d, k, p)

In [31]:
"""
Dynamic Programming to the rescue
"""
def min_coins(n, coins):

    dp = [float("inf")] * (n + 1)
    dp[0] = 0

    for i in range(1, n + 1):
        for c in coins:
            if c <= i:
                dp[i] = min(dp[i], dp[i - c] + 1)

    return dp[n]

In [32]:
def schedule_spt(processing_times):
    
    jobs = sorted(processing_times)
    
    current_time = 0
    completion_times = []
    
    for p in jobs:
        current_time += p
        completion_times.append(current_time)
    
    avg_completion = sum(completion_times) / len(completion_times)
    
    return jobs, completion_times, avg_completion

In [35]:
"""
Running shorter tasks first reduces the
waiting time for many tasks, which minimizes
the average completion time.

This is why the optimal strategy is to schedule jobs in
increasing order of processing time (Shortest Processing Time first).

| Customer | Processing Time |
| -------- | --------------- |
| A        | 10 minutes      |
| B        | 1 minute        |
| C        | 2 minutes       |


Processing time is the time required to serve a customer.
Completion time is the time when a customer finishes being served.

0 -----10-----11-----13
      A      B      C

Completion times:

C_A = 10
C_B = 11
C_C = 13

(10+11+13)/3 = 11.33

0--1--3---------13
   B  C      A

Completion times:

C_B = 1
C_C = 3
C_A = 13
(1+3+13)/3 = 5.67
"""
import heapq

def srpt_schedule(tasks):
    # tasks = [(release_time, processing_time)]

    tasks.sort()
    n = len(tasks)

    pq = []  # min heap by remaining time
    time = 0
    i = 0
    completion = []

    while i < n or pq:

        while i < n and tasks[i][0] <= time:
            release, p = tasks[i]
            heapq.heappush(pq, [p, release])
            i += 1

        if not pq:
            time = tasks[i][0]
            continue

        remaining, release = heapq.heappop(pq)

        remaining -= 1
        time += 1

        if remaining > 0:
            heapq.heappush(pq, [remaining, release])
        else:
            completion.append(time)

    avg_completion = sum(completion) / len(completion)
    return completion, avg_completion